In [1]:
import pandas as pd

df = pd.read_csv("tickets.csv")   # change path if needed
print("Rows:", df.shape)
print(df.columns)


Rows: (47837, 2)
Index(['Document', 'Topic_group'], dtype='object')


In [2]:
df["clean_text"] = (
    df["Document"]
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

df = df.dropna(subset=["clean_text", "Topic_group"])
df = df[df["clean_text"].str.len() > 30]

print("After filter:", df.shape)
print(df["Topic_group"].value_counts())


After filter: (47590, 3)
Topic_group
Hardware                 13539
HR Support               10853
Access                    7060
Miscellaneous             7039
Storage                   2772
Purchase                  2460
Internal Project          2114
Administrative rights     1753
Name: count, dtype: int64


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    df["clean_text"],
    df["Topic_group"],
    test_size=0.2,
    stratify=df["Topic_group"],
    random_state=42
)

print("Train:", len(X_train))
print("Val:", len(X_val))


Train: 38072
Val: 9518


In [4]:
from gensim.utils import simple_preprocess

X_train_tokens = X_train.apply(simple_preprocess)
X_val_tokens   = X_val.apply(simple_preprocess)


In [5]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences=X_train_tokens,
    vector_size=300,
    window=5,
    min_count=2,
    workers=4,
    sg=1   # skip-gram
)


In [6]:
import numpy as np

def sentence_vector(tokens, model):
    vectors = [
        model.wv[word]
        for word in tokens
        if word in model.wv
    ]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)
    
X_train_vec = np.vstack([
    sentence_vector(tokens, w2v_model)
    for tokens in X_train_tokens
])

X_val_vec = np.vstack([
    sentence_vector(tokens, w2v_model)
    for tokens in X_val_tokens
])


In [7]:
from sklearn.svm import LinearSVC

svm = LinearSVC()
svm.fit(X_train_vec, y_train)


LinearSVC()

In [8]:
from sklearn.metrics import classification_report, accuracy_score

y_pred = svm.predict(X_val_vec)

print("Accuracy:", accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred))


Accuracy: 0.8169783567976465


KeyboardInterrupt: 

In [4]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

X_train_emb = embedder.encode(
    X_train.tolist(),
    batch_size=32,
    show_progress_bar=True
)

X_val_emb = embedder.encode(
    X_val.tolist(),
    batch_size=32,
    show_progress_bar=True
)


Batches:   0%|          | 0/1190 [00:00<?, ?it/s]

Batches:   0%|          | 0/298 [00:00<?, ?it/s]

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

clf = LogisticRegression(max_iter=3000, n_jobs=-1)
clf.fit(X_train_emb, y_train)

y_pred = clf.predict(X_val_emb)

print(classification_report(y_val, y_pred))


                       precision    recall  f1-score   support

               Access       0.84      0.85      0.85      1412
Administrative rights       0.85      0.64      0.73       350
           HR Support       0.80      0.80      0.80      2171
             Hardware       0.75      0.81      0.78      2708
     Internal Project       0.85      0.76      0.80       423
        Miscellaneous       0.73      0.72      0.73      1408
             Purchase       0.95      0.83      0.89       492
              Storage       0.84      0.79      0.81       554

             accuracy                           0.79      9518
            macro avg       0.83      0.77      0.80      9518
         weighted avg       0.79      0.79      0.79      9518



In [7]:
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

svm = LinearSVC()
svm.fit(X_train_emb, y_train)

y_pred_svm = svm.predict(X_val_emb)

print(classification_report(y_val, y_pred_svm))


                       precision    recall  f1-score   support

               Access       0.85      0.86      0.85      1412
Administrative rights       0.86      0.64      0.73       350
           HR Support       0.79      0.79      0.79      2171
             Hardware       0.76      0.80      0.78      2708
     Internal Project       0.85      0.78      0.81       423
        Miscellaneous       0.73      0.73      0.73      1408
             Purchase       0.94      0.85      0.89       492
              Storage       0.83      0.82      0.83       554

             accuracy                           0.79      9518
            macro avg       0.83      0.78      0.80      9518
         weighted avg       0.80      0.79      0.79      9518



In [ ]:
    from sklearn.metrics import accuracy_score
print(accuracy_score(y_val, y_pred))
print(accuracy_score(y_val, y_pred_svm))

0.7916579113259088
0.7940743853750788


In [13]:
from sklearn.model_selection import train_test_split

df_core, df_rest = train_test_split(
    df,
    train_size=5000,   # adjust to 3000–5000 if needed
    stratify=df["Topic_group"],
    random_state=42
)

print("Core:", df_core.shape)
print("Remaining:", df_rest.shape)
print(df_core["Topic_group"].value_counts())

df_core.to_csv("core_tickets_v1.csv", index=False)
df_rest.to_csv("remaining_tickets.csv", index=False)


Core: (5000, 3)
Remaining: (42590, 3)
Topic_group
Hardware                 1423
HR Support               1140
Access                    742
Miscellaneous             740
Storage                   291
Purchase                  258
Internal Project          222
Administrative rights     184
Name: count, dtype: int64
